# Lesson 01 - Introduktion til AI-agenter

Velkommen til den første lektion i kurset **AI Agents for Beginners**!

En **AI-agent** er et program, der bruger en stor sprogmodel (LLM) som sin ræsonneringsmotor og kan tage *handlinger* i den virkelige verden — ringe til API'er, forespørge databaser eller køre kode — for at opnå et mål på vegne af en bruger.

I denne notesbog vil du bygge din første agent: en **Rejseagent**, som anbefaler feriedestinationer. Undervejs vil du lære hvordan du:

1. Forbinder til Azure AI Foundry Agent Service ved hjælp af **Microsoft Agent Framework**.
2. Giver agenten et **værktøj** — en almindelig Python-funktion, som den kan kalde.
3. Kører agenten og undersøger dens svar.
4. Streamer agentens svar token-for-token.


## Opsætning

Før du kører denne notesbog, skal du sørge for, at du har:

1. **Et Azure AI Foundry-projekt** med en implementeret chatmodel (f.eks. `gpt-4o-mini`).
2. **Logget ind med Azure CLI** — kør `az login` i din terminal.
3. **Satte de nødvendige miljøvariabler:**
   - `AZURE_AI_PROJECT_ENDPOINT` — dit Azure AI Foundry-projekts slutpunkt.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — navnet på din implementerede model.

Cellen nedenfor installerer de Python-pakker, du skal bruge.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Oprettelse af din første Agent

En agent har brug for to ting:

- **Instruktioner** der fortæller den *hvem den er* og *hvordan den skal opføre sig* (en systemprompt).
- **Værktøjer** — Python-funktioner dekoreret med `@tool`, som agenten kan kalde for at hente information eller udføre handlinger.

Nedenfor definerer vi et simpelt værktøj, der returnerer en liste over populære feriedestinationer. Agenten vil bruge dette værktøj, når en bruger spørger om rejseanbefalinger.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Streaming-svar

For en mere interaktiv oplevelse kan du **streame** agentens svar. I stedet for at vente på hele svaret, afleverer agenten tekststykker, efterhånden som de genereres. Dette er især nyttigt i chatgrænseflader, hvor du ønsker at vise output i realtid.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Summary

I denne lektion lærte du, hvordan du:

- **Opretter en udbyder** der forbinder til Azure AI Foundry Agent Service via `AzureAIProjectAgentProvider`.
- **Definerer et værktøj** ved hjælp af `@tool` dekoratøren, så agenten kan kalde dine Python-funktioner.
- **Kører agenten** med en brugerbesked og udskriver dets svar.
- **Strømmer svar** for output i realtid.

I den næste lektion vil vi udforske agentiske rammer mere grundigt og lære, hvordan man giver agenter mere kraftfulde værktøjer og evner til flertrins ræsonnering.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokument er blevet oversat ved hjælp af AI-oversættelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selvom vi bestræber os på nøjagtighed, skal du være opmærksom på, at automatiserede oversættelser kan indeholde fejl eller unøjagtigheder. Det oprindelige dokument på dets modersmål bør betragtes som den autoritative kilde. For vigtig information anbefales professionel menneskelig oversættelse. Vi påtager os intet ansvar for misforståelser eller fejltolkninger, der måtte opstå som følge af brugen af denne oversættelse.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
